# Snippet & Lexicon-Based Sentiment Analysis

This notebook implements a lexicon-based snippet approach for ESG sentiment analysis.
It generates rule-based sentiment scores using ESG-specific keywords, negation handling,
and intensifier detection. The outputs are saved for later hybrid fusion with transformer results.

In [13]:
# 1. Setup
import json
import pandas as pd
df = pd.read_csv("../data/processed/distilbert_baseline_5class.csv")

def lexicon_to_5class(score, high=200, low=50):
    if score >= high:
        return "VERY POSITIVE"
    elif score >= low:
        return "POSITIVE"
    elif score <= -high:
        return "VERY NEGATIVE"
    elif score <= -low:
        return "NEGATIVE"
    else:
        return "NEUTRAL"



# Load lemmatized corpus (created in preprocessing)
with open("../data/cleaned_v2/preprocessed_with_lemma.jsonl", "r") as f:
    corpus = [json.loads(line) for line in f]

print(f"✅ Loaded lemmatized corpus | docs: {len(corpus)}")

✅ Loaded lemmatized corpus | docs: 9


In [14]:
import re

# === 2. Custom ESG Lexicon + Scorer ===
ESG_LEXICON = {
    "positive": [
        "sustainable", "renewable", "green", "inclusive", "responsible",
        "net", "zero", "diversity", "environmental", "governance", "social",
        "ethical", "recycling", "efficiency", "compliance", "innovation",
        "equity", "fairness", "biodiversity", "community", "wellbeing"
    ],
    "negative": [
        "emission", "emissions", "pollution", "scandal", "deforestation",
        "fine", "controversy", "risk", "hazard", "lawsuit", "waste",
        "shortage", "violation", "fraud", "breach", "exploitation",
        "child", "forced", "toxic", "unethical"
    ]
}

NEGATIONS = ["no", "not", "never", "none", "without"]
INTENSIFIERS = ["very", "highly", "extremely", "significantly"]

def calculate_esg_lexicon_score(sentence: str) -> float:
    words = re.findall(r"\w+", sentence.lower())
    score = 0
    count = 0
    
    for i, word in enumerate(words):
        multiplier = 1.0
        if i > 0 and words[i-1] in INTENSIFIERS:
            multiplier = 1.5
        if word in ESG_LEXICON["positive"]:
            score += 1.0 * multiplier
            count += 1
        if word in ESG_LEXICON["negative"]:
            score -= 1.0 * multiplier
            count += 1
        if i > 0 and words[i-1] in NEGATIONS:
            score *= -1
    
    if count > 0:
        score = max(min(score / count, 1.0), -1.0)
    else:
        score = 0.0
    return score

print("✅ Custom ESG Lexicon Scorer loaded")

✅ Custom ESG Lexicon Scorer loaded


In [15]:
# 3. Apply Custom ESG Lexicon Scorer
df["lexicon_score"] = df["sentence"].apply(calculate_esg_lexicon_score)

print("✅ Lexicon scores added to DataFrame")
desc = df["lexicon_score"].describe()
print(desc.to_string())

✅ Lexicon scores added to DataFrame
count    14166.000000
mean         0.046292
std          0.549703
min         -1.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000


In [16]:
# 5-class mapping için önce normalize edilmiş lexicon_score'u kullan
df['lexicon_5class'] = df['lexicon_score'].apply(lambda x: lexicon_to_5class(x, high=0.5, low=0.05))

# Save new CSV with 5-class column
df.to_csv("../data/processed/snippet_scores_5class.csv", index=False)
print("✅ Saved with 5-class column")

# Hızlıca sınıf dağılımını kontrol et
print(df['lexicon_5class'].value_counts().to_string())


✅ Saved with 5-class column
lexicon_5class
NEUTRAL          9472
VERY POSITIVE    2506
VERY NEGATIVE    1863
POSITIVE          198
NEGATIVE          127
